<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 4 — Bitemporal loading and Airflow orchestration

<small>Data Creators › Day 4 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

<a id="how-to-use"></a>

## About this notebook

- Day 4's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [ ]:
# Setup: the demo environment on PATH, DuckDB and pandas for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day4-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day4-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a DuckDB file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 4 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

## Today

Morning: history (bitemporal loading) and orchestration (Airflow) with a long practical. Afternoon: the deployment picture, then two short exercises that turn a failed run into a decision and a note. Close by checking Friday's starting line.

| # | Section | What happens |
|---|---|---|
| 1 | **[Bitemporal loading](#s1)** | Demo: two clocks, three loads, four questions · Guest or volunteer: a real table · Shareback: which clock? |
| 2 | **[Airflow orchestration](#s2)** | Orientation: the DAG, and Friday · Break · Demo: the stack, a run, the UI, a controlled failure · Challenge 10: Airflow practical · Lunch · Shareback: one run, one log, one output check |
| 3 | **[CI/CD and release operations](#s3)** | Demo: GitOps, Harness, Datadog · Break · Activity 10: release decision · Activity 11: explain a failure and repair · Friday preparation |

<a id="s1"></a>

## 1. Bitemporal loading

### Demo — two clocks, three loads, four questions

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">The source corrects Monday's value on Wednesday. What was true on Monday, and what did we believe on Tuesday?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Two clocks on every row: business time (when it was true) and system time (when the warehouse knew it). Nothing is overwritten; corrections add versions.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Audit, reproducible reports and 'as was' questions all need the second clock; a status flag or an overwrite loses it.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Three loads on a tiny associate table: a day-1 file, a day-2 file with a change and a defect, a corrected re-delivery. Then four queries.</div></div></div>

**🖊️ Whiteboard — loading strategies and two clocks**

- How can new data land in a table?
- What are the two clocks?
- Full, incremental, balance?
- Why does a correction matter?
- UTC or local?

![One associate across three loads: four rows, nothing overwritten](../common/diagrams/png/d4-bitemporal-rows.png)

- Day 2's changed record closes the day-1 row's system clock and carries it forward with its business clock ended, then inserts the new version.
- The corrected re-delivery for the same business date closes only the system clock of the defective row and inserts the corrected one.
- As of Monday with today's knowledge: the carried-forward row. As was on Tuesday: the defective row. As is: the corrected row.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; three loads on DuckDB, then the four questions</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Three loads of a tiny associate feed, then the four questions, from one script.</div></div>

1. Run the cell: `demos/bitemporal/demo.sh` seeds, loads day 1, loads day 2, loads the correction, and prints the history and the as-is, as-of and as-was queries.
2. Read the history row by row: which load wrote each row, and which clock closed it.
3. Run the two `sql()` cells: the audit trail for one associate, then as-of 2 November.
4. Open `demos/bitemporal/macros/bitemporal_load.sql` (the cell prints it): three steps on one screen.

In [ ]:
!bash demos/bitemporal/demo.sh 2>&1 | sed -n '/-- Full history/,$p'

In [ ]:
sql("""select associate_id, job_title, tenure_months, dl_business_start_dt, dl_business_end_dt, dl_system_start_ts, dl_system_end_ts, dl_current_ind
          from training_demo.dim_associate where associate_id = 1001 order by dl_system_start_ts, dl_business_start_dt""", HERE / "demos/bitemporal/warehouse/bitemporal.duckdb")
# The audit trail for one associate. Point at each row: which load wrote it, and which clock closed it.

In [ ]:
sql("select associate_id, job_title, tenure_months from training_demo.dim_associate where dl_business_start_dt <= date '2026-11-02' and coalesce(dl_business_end_dt, date '9999-12-31') >= date '2026-11-02' and dl_system_end_ts is null order by 1", HERE / "demos/bitemporal/warehouse/bitemporal.duckdb")
# AS OF 2 November with today's knowledge: business clock covers the date, system clock open.

In [ ]:
!sed -n '1,40p' demos/bitemporal/macros/bitemporal_load.sql
# The whole strategy fits on a screen: close the system clock, carry the superseded business version forward, insert the new rows.

| The question | Which clock | The filter |
|---|---|---|
| What is current? | both open | `dl_current_ind` |
| What was true on date D, as we know it today? | business covers D, system open | `dl_business_start_dt <= D and coalesce(dl_business_end_dt, '9999-12-31') >= D and dl_system_end_ts is null` |
| What did the report say at time T? | system covers T | `dl_system_start_ts <= T and coalesce(dl_system_end_ts, '9999-12-31') > T` |
| Which versions did a correction replace? | same business start, later system start | order by `dl_system_start_ts` within the key |

### Guest or volunteer — a real bitemporal table

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 GUEST OR VOLUNTEER BRIEF — ten minutes</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Show one real bitemporal table, then take two questions.</div></div>

- The DL columns, and one row's history ordered by system start.
- One change that created a version; one correction if you have one.
- Which strategy the model uses: full, incremental or balance.
- If a defect once created a version, show it: the table is an audit of what the code did.

### Shareback — which clock?

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 SHAREBACK — which clock?</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Each table takes one question: say the clock, then the filter.</div></div>

- What was the associate's title on 2 November?
- What did the dashboard show on the morning of day 2?
- When did we learn about the correction?
- Which rows does an auditor need to see?

<a id="s2"></a>

## 2. Airflow orchestration

### Orientation — the DAG, and Friday's preview

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Yesterday's project builds when you type dbt build. Who types it at six in the morning?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Orchestration: deciding when the pipeline's parts run, in what order, with retries, logs and alerts.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>A product's promises about delivery time and freshness are kept by the scheduler, not by the SQL.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Read a DAG as a drawing; then the demo stack runs Day 3's project on a schedule.</div></div></div>

![The challenge's DAG: start, a freshness gate, the dbt task group, publish, end, with callbacks](../common/diagrams/png/d4-dag.png)

- The freshness gate runs first so a stale feed fails the run before any compute is spent.
- Cosmos renders the dbt project as one task per model and test, in dbt's own order.
- Callbacks live in configuration, not in the DAG's work: on failure, on retry, on SLA miss.
- The schedule is a cron in a time zone; catchup is off; retries have backoff.

In [ ]:
!sed -n '/^with DAG/,/^) as dag:/p' demos/airflow_local/dags/eod_prices/dags/eod_prices_daily.py
!grep -n '>>' demos/airflow_local/dags/eod_prices/dags/eod_prices_daily.py
# Day 2's generated product, orchestrated: the schedule is the contract's 07:30 ET commitment, 06:30 weekdays New York. Read >> as 'and then'.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 10:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Demo — the stack, a run, the UI, a controlled failure

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">What is actually running when you type astro dev start?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A DAG processor, a scheduler, workers, a triggerer, an API server with the UI, and a metadata database: on your laptop as containers, or hosted by Astronomer.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Knowing the pieces explains the symptoms: an import error is the processor; a queued task is the scheduler; a failed task is a worker log.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Trigger a run, read the graph and the grid, open a task log, read the code tab, break it on purpose, fix it.</div></div></div>

![Airflow's components, locally and hosted](../common/diagrams/png/d4-airflow-components.png)

- The DAG processor parses dags/ and reports import errors; the scheduler decides what is due; workers run tasks; the triggerer handles deferred waits; the API server serves the UI.
- astro dev start runs all of them as containers on the laptop; a hosted Astro deployment is the same thing run for you and vended per team.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the stack, a run, the UI</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Trigger a run on the local Astro stack and read it in the UI the way learners will.</div></div>

1. Run the cells below: seed, trigger, wait, status.
2. Open http://localhost:8080: the DAGs list with owner, schedule and tags.
3. Graph view: five boxes and the dbt task group.
4. Grid view: one column per run.
5. Click `publish_to_catalog` and open its log: the published rows are in it.
6. Code tab: the DAG file as deployed.

In [ ]:
!bash demos/airflow_local/run.sh seed 2>&1 | grep -E 'Done|rror'

In [ ]:
!bash demos/airflow_local/run.sh trigger && bash demos/airflow_local/run.sh wait

In [ ]:
!bash demos/airflow_local/run.sh status | tail -30

In [ ]:
!sed -n '/^default_args/,/^}/p' demos/airflow_local/dags/portfolio_analytics/dags/portfolio_analytics_daily.py
!grep -n 'def on_' demos/airflow_local/dags/portfolio_analytics/dags/callbacks.py
# default_args are inherited by every task: retries with backoff, the failure and retry callbacks. The SLA-miss callback sits on the DAG.

![Astro blueprints: YAML in, a DAG out](../common/diagrams/png/d4-blueprints.png)

- A blueprint is Python written once by the platform team with placeholders; a dataset's YAML fills them; the result is a registered DAG that triggers a Databricks job.
- Over ninety percent of products fit a blueprint; a custom DAG is for the rest, and can become a new blueprint.
- The challenge is Python on purpose: once the Python makes sense, the YAML is just configuration.

In [ ]:
!sed -n '1,30p' ../../reference/DataCreators-client/dev-container/edo-training-data-products/data_products/.blueprints/dag_exec_check.py
# A platform blueprint from the vended repository: the Python learners never write, filled from YAML.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; a controlled failure, then the repair</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Break the freshness gate on purpose, read the log, repair, rerun.</div></div>

1. Run the stale cell: the custodian feed looks two days old; a run is triggered.
2. Watch the freshness gate fail and everything downstream go `upstream_failed`: no publish, no rows.
3. Open the failed task's log in the UI and read the first useful line aloud.
4. Run the fresh cell: the feed is stamped fresh, a run is triggered, green.

In [ ]:
!bash demos/airflow_local/run.sh stale && bash demos/airflow_local/run.sh trigger && bash demos/airflow_local/run.sh wait && bash demos/airflow_local/run.sh status | grep -E 'source_freshness|publish|state' | head -6

In [ ]:
!bash demos/airflow_local/run.sh fresh && bash demos/airflow_local/run.sh trigger && bash demos/airflow_local/run.sh wait

### Challenge 10 — Airflow practical

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; Airflow practical (challenge 03) &nbsp;·&nbsp; until 13:30, with lunch</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">Wire, schedule and run the Portfolio Analytics DAG with Cosmos; Phase 2 is production operations. &nbsp;·&nbsp; <code>challenges/v1/03-airflow-orchestration/README.md</code></div></div>

**Prerequisites, in this order**

1. `mise run setup:databricks` done, in a terminal where `echo $DBT_CATALOG` prints a value.
2. `(cd challenges/v1/02-dbt-pipeline/dbt && dbt seed)`: the DAG reads the same four feeds, refreshed.
3. From the repo root, `astro dev start`, only after setup finished; the UI address is the one astro prints, not a fixed localhost:8080.

**Phase 1, about 1.5 hours**

1. 1 Why orchestration.
2. 2 Anatomy of a DAG.
3. 3 Connect to Databricks: `dbt debug`.
4. 4 The freshness gate: `astro dev run tasks test portfolio_analytics_daily source_freshness_check 2026-05-04`.
5. 5 The Cosmos `DbtTaskGroup`.
6. 6 Wire the chain, the one TODO: `start >> source_freshness_check >> dbt_pipeline >> publish_to_catalog >> end`.
7. 7 Scheduling: a timezone-aware cron.
8. 8 Retries and exponential backoff.
9. 9 First full run.
10. 10 Reading logs and re-running.

**Phase 2**

1. 1 The vendor-neutral alert transport.
2. 2 Tiered failure alerting.
3. 3 The P2 and P3 tiers: SLA miss and retry.
4. 4 Wire the callbacks and the SLA into the DAG.
5. 5 The freshness gate as a contract.
6. 6 The monitoring config.
7. 7 The runbook and the consumer guide.
8. 8 Structural tests for the DAG.
9. 9 Stretch: the `DatabricksSubmitRunOperator` variant.
10. 10 Capstone: a production-ready DAG.

**Gate.** `mise run challenge:verify 03` is a static check of the wiring; the live run is the evidence: graph, logs, states.

**Finished early?** What happens if the input is late? If the quality check fails? If the run is repeated? Show a log or output check for each answer.

| Symptom | Cause | Fix |
|---|---|---|
| `dags list-import-errors` shows a Cosmos error ending in an empty catalog name | Astro started before Databricks setup wrote the env | New terminal where `echo $DBT_CATALOG` prints a value, then `astro dev restart --wait 5m` |
| The freshness gate fails on a fresh seed | Connection error, not staleness: the task now says which | Same cold-start fix; then re-run the task |
| `airflow: command not found` | The CLI lives in the containers | Prefix with `astro dev run` |
| `astro dev parse` says 0 DAGs | It does not mount data_products | Use the README's `python -c "import ast…"` check and `astro dev run dags list-import-errors` |
| A BashOperator DAG is rejected | CG's cluster policy blocks arbitrary-shell operators | PythonOperator calling subprocess, as the starter does |
| Tasks run one at a time | `max_active_tasks` (learners: 4; this laptop: 1 for DuckDB) | Expected; explain why |
| `dbt docs serve` fails to bind | `DBT_HOST` is the Databricks host | `--host 127.0.0.1 --port 8081` |
| Two Astro environments on one laptop | Another project is running | `docker ps` and look for the devcontainer's scheduler |

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Shareback — one run, one log line, one output check per pair

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 SHAREBACK — 13:30, one pair at a time</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Show the run ID, the log line that told you what happened, and the table it produced.</div></div>

- Keep credential and path errors separate from the intended data-quality failure.
- Fast finishers: what happens if the input is late? If the quality check fails? If the run is repeated?

<a id="s3"></a>

## 3. CI/CD and release operations

### Demo — GitOps, Harness, Datadog

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Your DAG and dbt project are in Git. How do they get to production, and who presses the button?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Continuous integration (merge often) and continuous deployment (release often), driven by GitHub events through Harness, into Astro and Databricks, watched by Datadog.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Creators rarely operate this layer, but they read it: which pipeline ran, where the logs are, who deployed last night.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>The GitOps picture; a Harness walkthrough (guest, screenshots, or the anchor); monitors as the SLIs made visible.</div></div></div>

![From a branch to production: pull request, merge, Harness, SQA, production with a change record](../common/diagrams/png/d4-cicd.png)

- Integration is merging; continuous means daily or hourly, not monthly. Deployment is releasing to an environment.
- A merge fires a webhook; Harness builds, tests and deploys; the SQA environment is shared, so check who deployed last; production needs a change record.
- GitOps: the Git history becomes the history of what was live. CG is 'halfway': events trigger pipelines, with an approval step between.

**🖊️ Whiteboard — CI/CD**

- What do CI and CD mean?
- Why bother?
- What are the downsides?
- What is Harness here?

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Harness, with screenshots or a guest</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Walk a deployment to SQA and then to production: live if someone has access, else from the pre-flight screenshots.</div></div>

1. Pick the environment and find the repository's pipeline.
2. SQA: Run, apply the SQA input set, choose the feature branch, run; a few minutes.
3. Check who deployed to SQA last and whether they finished testing: SQA is shared.
4. Production: the PRD pipeline, fixed values, the release branch, and a change record (CRQ) number before anything runs.
5. Open the logs of a failed execution.

In [ ]:
!sed -n '1,40p' demos/airflow_local/dags/portfolio_analytics/include/monitoring/portfolio_analytics.yml
# The challenge's monitoring config: six panels, each an SLI with an alert condition and a channel. This is what a Datadog monitor is for: the promises from Day 2, watched.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 14:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Activity 10 — release decision

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 10 &nbsp;—&nbsp; Release decision &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 20 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>The nightly build produced a candidate output with 500 rows but only 497 distinct IDs, so the uniqueness check failed and the candidate was not published. Yesterday's accepted output is still available to consumers. In ten minutes, decide what should happen next and draft a two-sentence update to the consumer. Then ten minutes of discussion.</strong><br><br><strong>Produce:</strong><br>• A decision: what is released, what stays blocked, and who decides<br>• A two-sentence update to the consumer<br>• The information you are missing and who can supply it<br><br><strong>Debrief questions:</strong><br>• Does the consumer's freshness commitment allow yesterday's output to stand today?<br>• What would make the three duplicate IDs a data problem rather than a pipeline problem?<br>• Who gets to override a failed quality gate, and how is that recorded?</div></div>

<details><summary><strong>Discussion notes for Activity 10</strong></summary>

- The failed candidate stays blocked: three IDs appear twice, and until someone knows why, publishing it would publish a lie about uniqueness.
- Yesterday's accepted output is acceptable only if the consumer's freshness commitment allows a one-day-old sheet; the contract says, not the engineer.
- Missing information: which three IDs, whether the source re-delivered, whether the dedup rule or the source changed, who owns the source. The consumer update names what is known, what is not, and when the next update comes.
- Who overrides a failed gate, and where that decision is recorded, is a governance question worth leaving open in the room.

</details>

### Activity 11 — explain a failure and repair

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 11 &nbsp;—&nbsp; Explain a failure and repair &nbsp;·&nbsp; Pairs, then swap &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Write a short note about the Airflow failure and repair you ran today (or the prepared example if your environment was blocked): the affected run, the useful error, the cause, the action taken, the output check, and anything still unresolved. Swap notes with another pair for five minutes: could they locate the run and explain what happened from the note alone?</strong><br><br><strong>Produce:</strong><br>• The note, using the template below<br>• The other pair's verdict: could they find the run?<br><br><strong>Debrief questions:</strong><br>• Which line of the note was the most useful to the reader?<br>• What did you have to go back to the UI to find?</div></div>

<details><summary><strong>Prepared failure example</strong> (for pairs whose environment was blocked)</summary>

Expected: the freshness gate fails because the custodian feed is older than its `error_after` threshold; `dbt_pipeline` and `publish_to_catalog` are `upstream_failed`; no rows are published; the previous run's output stands. Observed in the demo stack: the `source_freshness_check` log ends with "dbt source freshness failed: the custodian feed breached its error_after threshold"; the repair was to refresh the feed and trigger a new run, which published eight rows. Distinguish this from a credentials or path error, whose first useful line names a missing environment value or executable.

</details>

### Friday preparation — the checklist, then the close

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 FRIDAY CHECKLIST — every learner, before leaving</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Five checks; any failure goes on the list with a name and a time it will be fixed.</div></div>

- ☐ Open `challenges/v1/04-atlas-capstone/README.md` and read the plan of the day.
- ☐ From the repo root: `test -f .mise.toml && test -d data_products/atlas_elt && echo OK`.
- ☐ `mise exec -- databricks auth describe --profile DEFAULT`.
- ☐ From `data_products/atlas_elt`: `dbt debug` passes.
- ☐ `astro dev stop`, so tomorrow starts clean.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ The DAG, run and task identities, useful logs, an output check and a short explanation of the failure and repair, per pair.<br>☐ Every learner has opened challenges/v1/04-atlas-capstone/README.md and run the readiness checks in data_products/atlas_elt.<br>☐ A list of setup issues that could stop Friday's start, each with the person responsible for the next action.<br>☐ The Airflow stack stopped on the learners' laptops (astro dev stop) so Friday starts clean.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>